# Smart Semsar  — End-to-End Speech & Intelligence Pipeline
### Fixed Solutions — Speech On Real Calls Benchmark
---
This notebook implements the complete speech intelligence pipeline and benchmark:
1. **ASR Evaluation (WER & CER)** on noisy Egyptian Arabic phone audio.
2. **Speaker Separation & Role Classification** (Broker vs. Client accuracy & confusion matrix).
3. **VAD, Diarization, Alignment, and Structured Customer Profiling**.

## SECTION 0 — Environment & Configuration

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    RAW_CALLS_DIR, ANNOTATIONS_DIR, OUTPUT_DIR, REPORTS_DIR,
    TARGET_SAMPLE_RATE, BROKER_INDICATORS, CLIENT_INDICATORS
)
from src.text_normalization import normalize_arabic_text
from src.asr_evaluator import calculate_wer, calculate_cer, format_alignment_display
from src.role_separator import RoleSeparator
from src.diarization_evaluator import evaluate_der, evaluate_broker_client_separation
from src.audio_processing import load_wav_audio, compute_audio_quality_metrics

print("Project Root      :", PROJECT_ROOT)
print("Calls Directory   :", RAW_CALLS_DIR)
print("Reports Directory :", REPORTS_DIR)
print("Environment initialized successfully.")

Project Root      : d:\smart_semsar_speech_eval\smart_semsar_speech
Calls Directory   : D:\smart_semsar_speech_eval\smart_semsar_speech\data\real_estate_calls\audio
Reports Directory : D:\smart_semsar_speech_eval\smart_semsar_speech\data\outputs\reports
Environment initialized successfully.


## SECTION 1 — Audio Inspection & Quality Metrics

In [2]:
CALL_ID = "call_03"
audio_path = RAW_CALLS_DIR / f"{CALL_ID}.wav"

audio_data, sr = load_wav_audio(audio_path)
qc_metrics = compute_audio_quality_metrics(audio_data, sr)

print("=" * 50)
print("AUDIO QUALITY QC REPORT")
print("=" * 50)
for k, v in qc_metrics.items():
    print(f"{k:<25}: {v}")

AUDIO QUALITY QC REPORT
duration_seconds         : 14.5
sample_rate              : 16000
samples                  : 232000
mean_amplitude           : -3.143534195260145e-05
std_amplitude            : 0.11629026383161545
rms_energy               : 0.11629
peak_amplitude           : 0.373383
clipped_samples          : 0
clipping_ratio_pct       : 0.0000%
quality_status           : Clean


## SECTION 2 — Egyptian Arabic Text Normalization

In [3]:
raw_text = "مساءُ الخَيْرِ، لو سَمَحْتْ أَنَا كُنْتُ بَدَوَّرْ على شَقَّةٍ في نِيُو كَايْرُو... مِيزَانِيَّتِي ٥ مِلْيُون!"
normalized = normalize_arabic_text(raw_text)

print("Original Text   :", raw_text)
print("Normalized Text :", normalized)

Original Text   : مساءُ الخَيْرِ، لو سَمَحْتْ أَنَا كُنْتُ بَدَوَّرْ على شَقَّةٍ في نِيُو كَايْرُو... مِيزَانِيَّتِي ٥ مِلْيُون!
Normalized Text : مساء الخير لو سمحت انا كنت بدور علي شقه في نيو كايرو ميزانيتي 5 مليون


## SECTION 3 — ASR Error Rate Evaluation (WER & CER)

In [4]:
import json
annot_path = ANNOTATIONS_DIR / f"{CALL_ID}.json"
with open(annot_path, "r", encoding="utf-8") as f:
    call_annot = json.load(f)

ref_text = call_annot["reference_transcript"]
hyp_text = call_annot["hypothesis_transcript"]

wer_res = calculate_wer(ref_text, hyp_text, normalize=True)
cer_res = calculate_cer(ref_text, hyp_text, normalize=True)

print("=" * 60)
print("ASR ACCURACY ON CALL_00")
print("=" * 60)
print(f"Word Error Rate (WER)      : {wer_res['error_rate_pct']}")
print(f"Character Error Rate (CER) : {cer_res['error_rate_pct']}")
print(f"Word Accuracy              : {wer_res['accuracy_pct']}")
print(f"Substitutions: {wer_res['substitutions']} | Deletions: {wer_res['deletions']} | Insertions: {wer_res['insertions']} | Hits: {wer_res['hits']}")
print("Token Alignment:")
print(format_alignment_display(wer_res["alignment"]))

ASR ACCURACY ON CALL_00
Word Error Rate (WER)      : 0.00%
Character Error Rate (CER) : 0.00%
Word Accuracy              : 100.00%
Substitutions: 0 | Deletions: 0 | Insertions: 0 | Hits: 32
Token Alignment:
REF:  مساء  الخير  كنت  عايز  اعرف  تفاصيل  المكاتب  الاداريه   في  العاصمه  الاداريه  اهلا  بيك   يا  فندم  معاكي  ندي   من  شركه  سيلز  كابيتال   في  طرح  جديد  بمقدم  عشرين   في  الميه  وعائد  استثماري  علي  المقدم 
HYP:  مساء  الخير  كنت  عايز  اعرف  تفاصيل  المكاتب  الاداريه   في  العاصمه  الاداريه  اهلا  بيك   يا  فندم  معاكي  ندي   من  شركه  سيلز  كابيتال   في  طرح  جديد  بمقدم  عشرين   في  الميه  وعائد  استثماري  علي  المقدم 
OP :  HIT    HIT   HIT  HIT   HIT    HIT      HIT      HIT     HIT    HIT      HIT     HIT   HIT  HIT  HIT    HIT   HIT  HIT  HIT   HIT     HIT    HIT  HIT  HIT    HIT    HIT   HIT   HIT    HIT     HIT     HIT   HIT   


## SECTION 4 — Speaker Diarization & Role Separation (Broker vs. Client)

In [6]:
role_separator = RoleSeparator()
ref_turns = call_annot["reference_turns"]

# Separate roles
separation_result = role_separator.separate_roles(ref_turns)

print("=" * 60)
print("SPEAKER ROLE SEPARATION RESULT")
print("=" * 60)
print("Detected Roles Mapping:", separation_result["role_mapping"])
print(f"Separation Confidence : {separation_result['confidence'] * 100:.1f}%")
print("Conversation with Assigned Roles:")
for turn in separation_result["assigned_turns"]:
    role_title = "السمسار (Broker)" if turn["role"] == "broker" else "العميل (Client)"
    print(f"[{turn['start']:.2f}s → {turn['end']:.2f}s] {role_title}: {turn['text']}")

SPEAKER ROLE SEPARATION RESULT
Detected Roles Mapping: {'SPEAKER_01': 'broker', 'SPEAKER_00': 'client'}
Separation Confidence : 66.7%
Conversation with Assigned Roles:
[0.00s → 5.50s] العميل (Client): مساء الخير كنت عايز اعرف تفاصيل المكاتب الادارية في العاصمة الادارية
[5.50s → 14.50s] السمسار (Broker): اهلا بيك يا فندم معاكي ندي من شركة سيلز كابيتال في طرح جديد بمقدم عشرين في المية وعائد استثماري علي المقدم


## SECTION 5 — Role Separation Accuracy & Confusion Matrix

In [8]:
sep_eval = evaluate_broker_client_separation(
    reference_turns=ref_turns,
    predicted_turns=separation_result["assigned_turns"],
    total_duration=call_annot["duration"]
)

print("=" * 60)
print("BROKER VS. CLIENT SEPARATION ACCURACY")
print("=" * 60)
print(f"Call-Level Role Correct : {sep_eval['call_level_role_correct']}")
print(f"Time-Weighted Accuracy  : {sep_eval['time_weighted_separation_accuracy_pct']}")
print(f"Macro F1-Score          : {sep_eval['macro_f1_pct']}")
print(f"Broker F1-Score         : {sep_eval['broker_metrics']['f1_score_pct']}")
print(f"Client F1-Score         : {sep_eval['client_metrics']['f1_score_pct']}")
print("Confusion Matrix (Seconds):")
cm = sep_eval["confusion_matrix"]
print(f"Broker classified as Broker: {cm['broker_as_broker_sec']}s ({cm['broker_accuracy_pct']})")
print(f"Client classified as Client: {cm['client_as_client_sec']}s ({cm['client_accuracy_pct']})")

BROKER VS. CLIENT SEPARATION ACCURACY
Call-Level Role Correct : True
Time-Weighted Accuracy  : 100.00%
Macro F1-Score          : 100.00%
Broker F1-Score         : 100.00%
Client F1-Score         : 100.00%
Confusion Matrix (Seconds):
Broker classified as Broker: 9.0s (100.00%)
Client classified as Client: 5.5s (100.00%)


## SECTION 6 — Run Full Corpus Benchmark

In [9]:
from evaluate_asr import run_asr_noise_benchmark
from evaluate_diarization import run_diarization_separation_benchmark

asr_rep = run_asr_noise_benchmark(ANNOTATIONS_DIR, REPORTS_DIR / "nb_asr_report.json")
diar_rep = run_diarization_separation_benchmark(ANNOTATIONS_DIR, REPORTS_DIR / "nb_diar_report.json")
print("Corpus benchmark finished successfully.")

FIXED SOLUTIONS - SMART SEMSAR SPEECH BENCHMARK
Task: Word and Character Error Rates on Noisy Egyptian Arabic Phone Audio
Total Call Samples: 5

╒═══════════════════════════════════╤═══════╤═══════╤═════════════════╤═════════════════════╤═════════════════╤══════════════════╤════════════╕
│ Audio Condition                   │ WER   │ CER   │ Word Accuracy   │   Substitutions (S) │   Deletions (D) │   Insertions (I) │   Hits (H) │
╞═══════════════════════════════════╪═══════╪═══════╪═════════════════╪═════════════════════╪═════════════════╪══════════════════╪════════════╡
│ Clean Studio/Mobile               │ 8.24% │ 4.65% │ 91.76%          │                  12 │               1 │                2 │        169 │
├───────────────────────────────────┼───────┼───────┼─────────────────┼─────────────────────┼─────────────────┼──────────────────┼────────────┤
│ Telephony Filter (G.712 Bandpass) │ 1.10% │ 1.42% │ 98.90%          │                   0 │               2 │                0 │     